# Practice : Open-Source sLLM 실행 및 비교

[Task]
1. Hugging Face에서 sLLM을 직접 다운로드하여 실행
2. 동일 프롬프트에 대한 범용 LLM(ChatGPT, Claude 등) / Base 모델 / Instruct 모델의 응답 비교
3. 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 출력 확인

[Model]
- Base / Instruct 비교: Qwen2.5-1.5B, Qwen2.5-1.5B-Instruct
- 특화 튜닝 확인: SLM-SQL-0.5B

[Note]
- Colab 환경을 기본으로 작성, 로컬 환경 실행 시 안내를 별도 표기
- 모두 공개(gated 아님) 모델이므로 별도 로그인 불필요

## 1. Colab GPU 환경 설정 [Colab 전용]

- 런타임 > 런타임 유형 변경 > 하드웨어 가속기: T4 GPU 선택 후 저장
- 로컬 환경 사용자는 이 섹션 건너뛰기
- 무료 Colab은 세션 시간과 GPU 사용량에 제한이 있음. 연결이 끊기면 런타임을 다시 시작하고 위 셀부터 재실행

In [1]:
!nvidia-smi

Tue Aug 18 02:26:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU 사용 가능: True
Device: Tesla T4


## 2. 라이브러리 설치

- transformers 최신 버전 필요 (구버전 사용 시 모델 설정 인식 오류 발생 가능)
- 로컬 환경에서 이미 설치되어 있다면 생략 가능

In [3]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 50.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


## 3. 라이브러리 Import

In [4]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

# GPU마다 지원하는 연산 정밀도가 다름 (예: T4는 bfloat16 미지원, fp16만 가속됨)
# 사용 가능한 최적 dtype을 자동 선택
if torch.cuda.is_available():
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    compute_dtype = torch.float32

print("Transformers version:", transformers.__version__)
print("Compute dtype:", compute_dtype)

Transformers version: 5.15.0
Compute dtype: torch.bfloat16


[Note] 모델 다운로드 저장 위치
- Hugging Face 캐시 경로"~/.cache/huggingface/hub"에 저장됨
- Colab은 런타임(가상 인스턴스)의 로컬 디스크에 저장 : 런타임 종료 시 캐시도 함께 삭제되어 재접속 시 재다운로드 필요
- 로컬 환경은 컴퓨터에 캐시가 계속 남아있어 재실행 시 다운로드 없이 바로 로드됨

## 4. Base 모델 로드 (Qwen2.5-1.5B)

- Instruction Tuning이 적용되지 않은 사전학습 모델
- Chat Template 없이 순수 텍스트 이어쓰기 방식으로 동작

In [5]:
base_model_id = "Qwen/Qwen2.5-1.5B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)

print("Base 모델 로드 완료:", base_model_id)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base 모델 로드 완료: Qwen/Qwen2.5-1.5B


## 5. Instruct 모델 로드 (Qwen2.5-1.5B-Instruct)

- Instruction Tuning이 적용된 모델
- Chat Template을 적용하여 대화 형식으로 입력

In [6]:
instruct_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_id)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)

print("Instruct 모델 로드 완료:", instruct_model_id)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Instruct 모델 로드 완료: Qwen/Qwen2.5-1.5B-Instruct


## 6. 추론 함수 정의

- Base: 프롬프트를 그대로 이어쓰기
- Instruct: Chat Template 적용 후 생성

[Note] Chat Template ?
- Instruct 모델은 "<|im_start|>user ... <|im_end|>"처럼 발화자를 구분하는 특수 토큰 구조로 학습됨
- apply_chat_template()은 이 구조를 자동으로 만들어주는 함수이며, Base 모델은 이런 구조를 학습한 적이 없어 사용하지 않음

In [7]:
def generate_base(prompt: str, max_new_tokens: int = 200) -> str:
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    output_ids = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return base_tokenizer.decode(generated, skip_special_tokens=True)


def generate_instruct(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    # return_dict=True: transformers 버전에 따라 apply_chat_template이
    # 텐서 대신 BatchEncoding을 반환하는 경우가 있어 명시적으로 지정
    inputs = instruct_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(instruct_model.device)
    output_ids = instruct_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return instruct_tokenizer.decode(generated, skip_special_tokens=True)

## 7. 비교 프롬프트 세트

[Note]
- 범용 LLM(ChatGPT, Claude 등) 결과는 아래 프롬프트를 웹 UI에 직접 질의하여 별도로 기록
- 이 노트북에서는 sLLM(Base/Instruct)만 코드로 직접 비교

In [8]:
single_turn_prompts = {
    "1_사실질문": "세종대왕이 만든 문자는 무엇인가요?",
    "2_지시문형": (
        "다음 문장을 한 문장으로 요약해줘: "
        "'인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, "
        "특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'"
    ),
    "3_추론문제": (
        "한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, "
        "안경을 쓴 학생 중 여학생은 7명입니다. "
        "안경을 쓰지 않은 학생 중 남학생이 10명이라면, "
        "안경을 쓰지 않은 여학생은 몇 명인가요?"
    ),
}

for name, prompt in single_turn_prompts.items():
    print(f"[{name}] {prompt}\n")

[1_사실질문] 세종대왕이 만든 문자는 무엇인가요?

[2_지시문형] 다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'

[3_추론문제] 한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, 안경을 쓴 학생 중 여학생은 7명입니다. 안경을 쓰지 않은 학생 중 남학생이 10명이라면, 안경을 쓰지 않은 여학생은 몇 명인가요?



## 8. 단일 턴 프롬프트 실행 및 비교

[Note]
- Base 모델은 종료 시점을 스스로 판단하지 못해 매번 max_new_tokens만큼 끝까지 생성 : Instruct보다 시간이 더 걸림


In [9]:
# 실행 전 device 확인: cpu로 나오면 생성이 매우 느릴 수 있음
print("Base 모델 device:", base_model.device, "| dtype:", base_model.dtype)
print("Instruct 모델 device:", instruct_model.device, "| dtype:", instruct_model.dtype)

if base_model.device.type == "cpu":
    print("\n GPU가 아닌 CPU에서 실행 중")


Base 모델 device: cuda:0 | dtype: torch.bfloat16
Instruct 모델 device: cuda:0 | dtype: torch.bfloat16


In [10]:
for name, prompt in single_turn_prompts.items():
    print(f"=== {name} ===")
    print("[Prompt]", prompt)
    print("\n[Base]")
    print(generate_base(prompt))
    print("\n[Instruct]")
    print(generate_instruct(prompt))
    print("\n" + "=" * 60 + "\n")

=== 1_사실질문 ===
[Prompt] 세종대왕이 만든 문자는 무엇인가요?

[Base]
 세종대왕이 만든 문자는 '조선자유자치'입니다. 세종대왕은 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 그는 조선의 문화를 발전시키기 위해 '조선자유자치'라는 정책을 내렸습니다. 이 정책은 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 세종대왕은 조선의 문화를 발전시키기 위해 '조선자유자치'라는 정책을 내렸습니다. 이 정책은 조선의 문화를 발전시키고, 민주주의를 실현하기 위해 노력했습니다. 세종대왕은 조선의 문화를 발전시키기 위해 '조선자유자치'라는 정책을 내렸습니다. 이 정책은 조선

[Instruct]
세종대왕은 조선의 세 번째 왕으로서, 1418년부터 1450년까지 재위했습니다. 그는 많은 업적을 남겼지만, 가장 유명한 것은 한글을 발전시킨 것입니다.

세종대왕은 한글을 발전시키기 위해 노력했고, 이를 통해 한국어를 현대에 이르게 한 중요한 인물입니다. 그의 노력은 한글의 기초를 마련하고, 글자와 음절의 관계를 명확히 하였습니다. 이러한 노력 덕분에 오늘날 우리가 사용하는 한국어가 가능하게 되었습니다.

따라서, 세종대왕이 만든 문자는 바로 한글입니다.


=== 2_지시문형 ===
[Prompt] 다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'

[Base]
 이는 인공지능 기술이 기업의 경쟁력과 수익성에 미치는 영향을 살펴보는 데 도움이 될 것이다. 인공지능 기술의 발전은 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다. 이는 기업의 경쟁력과 수익성에 미치는 영향을 살펴보는 데 도움이 될 것이다.

다음 문장을 한 문장으로 요약해줘: '인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, 특히 제조업과 금융업에

## 9. 멀티턴 대화 비교

[Note]
- Base 모델은 대화 형식(Chat Template)을 학습하지 않았으므로, 직전 턴과 답변을 프롬프트에 이어붙이는 방식으로 실행
- Instruct 모델은 messages 리스트에 대화 히스토리를 누적하여 실행

In [11]:
turn1 = "제주도 여행 코스를 하나 추천해줘"
turn2 = "방금 그거, 1박 2일로 압축해줄 수 있어?"

# Base: 이전 턴과 응답을 프롬프트에 이어붙임
base_turn1_response = generate_base(turn1)
base_multiturn_prompt = f"{turn1}\n{base_turn1_response}\n{turn2}"
base_turn2_response = generate_base(base_multiturn_prompt)

print("[Base - Turn1]", base_turn1_response)
print("\n[Base - Turn2]", base_turn2_response)

[Base - Turn1] 요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도

[Base - Turn2]  제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 코스를 하나 추천해줘요. 제주도 여행 �


In [12]:
# Instruct: messages 리스트에 대화 히스토리 누적
messages = [{"role": "user", "content": turn1}]
inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn1_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

messages.append({"role": "assistant", "content": instruct_turn1_response})
messages.append({"role": "user", "content": turn2})

inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn2_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

print("[Instruct - Turn1]", instruct_turn1_response)
print("\n[Instruct - Turn2]", instruct_turn2_response)

[Instruct - Turn1] 제주도는 한국에서 가장 인기 있는 관광지 중 하나로, 다양한 명소와 풍경을 제공합니다. 아래는 제주도의 주요 관광지들을 기반으로 한 여행 코스입니다:

1. 제주시 - 제주국립공원 및 해변
2. 서귀포시 - 서귀포 섬과 해수욕장
3. 제주시 - 제주도 최고의 해수욕장인 '제주 해양마라톤'
4. 제주시 - 제주도의 중심부에 위치한 '제주도성'
5. 서귀포시 - 서귀포 섬의 자연美景과 해수욕장

이러한 곳들을 방문하여 제주도의 아름다움을 체험하세요. 또한, 제주도의 특별한 요리를 맛볼 수 있는 '제주도식당'이나 '제주도 레스토랑'을 찾아보세요. 이곳에서는 제주도의 전통 음식을 즐길 수 있습니다.

여행 계획은 개인의 취향과 예산에 따라 달라질 수 있으니, 적절한 정보를 얻어 여행 준비를 마무리하시기 바랍니다.

[Instruct - Turn2] 물론이죠! 1박 2일로 압축된 제주도 여행 코스를 소개해 드리겠습니다:

1. 제주시 - 제주국립공원 및 해변
   - 오전: 제주국립공원 입장료 지불 후, 공원 내 조산대 산책
   - 오후: 해변에서 바다 보호 정신 깨우기 활동 (예: 바다 건강 운동)

2. 서귀포시 - 서귀포 섬과 해수욕장
   - 오전: 서귀포 섬 탐사, 해수욕장을 방문
   - 오후: 서귀포 섬의 자연美景을 감상하고, 해수욕장에서 편안히 쉬거나 물놀이를 즐기기

3. 제주시 - 제주도 성
   - 오전: 제주도성 입장료 지불 후, 성 안내서를 참고로 성 안을 돌아보고
   - 오후: 제주도성 근처의 점심 식사를 즐기기

4. 서귀포시 - 서귀포 섬의 자연美景과 해수욕장
   - 오전: 서귀포 섬의 자연美景을 감상하고, 해수욕장을 방문하기
   - 오후: 서귀포 섬의 자연美景을 다시 한번 감상하거나, 해수욕장에서 편안히 쉬거나 물놀이를 즐기기

이렇게 하면 1박 2일 동안 제주도의 아름다움을 체험할 수 있을 것입니다. 각각의 장소마다 특별한 경험을 나누며, 제주도의 매력에 더욱 집중하실 수 있습니다.


## 10. 범용 LLM과의 비교 기록

[Task]
- 위 프롬프트(1~4)를 ChatGPT, Claude 등 범용 LLM 웹 UI에 동일하게 질의
- 아래 표에 결과를 요약하여 정리

| 프롬프트 | 범용 LLM | sLLM-Base | sLLM-Instruct |
|---|---|---|---|
| 1. 사실 질문 | 세종대왕이 만든 문자는 훈민정음입니다. 오늘날의 한글을 뜻합니다. | 정답(△), 문장이 중간에 끊기거나 반복됨 | 정답(O), 간결하고 핵심적인 답변 |
| 2. 지시문형 | 인공지능 발전으로 산업 자동화가 가속화되며, 특히 제조업과 금융업에서 변화가 두드러지고 있다. | 지시 무시, 프롬프트 텍스트를 이어쓰는 경향 | 지시사항 대부분 준수 (복잡한 조건은 일부 누락) |
| 3. 추론 문제 | 안경을 쓰지 않은 학생은 (30-12=18)명입니다.그중 남학생이 10명이므로, 안경을 쓰지 않은 여학생은 18-10=8명입니다. | 추론 실패, 엉뚱한 텍스트 생성 | 기본 추론 가능, 복잡한 연산에서 논리적 오류 발생 |
| 4. 멀티턴 | 렌터카 기준 제주 동부 1일 코스를 추천합니다.비자림(오전) → 성산일출봉 → 성산에서 점심 → 섭지코지 → 광치기해변(노을)숲·오름·해안을 하루에 고르게 즐기면서 이동 동선도 비교적 짧습니다. 비가 많이 오면 성산일출봉 대신 아쿠아플라넷 같은 실내 관광지로 바꾸세요.방문 전 운영·기상 정보는 비짓제주 공식 관광 가이드에서 확인하는 것이 좋습니다. | 문맥 파악 불가, 독립적인 문장 생성 | 단기 문맥 유지 가능, 대화가 길어지면 맥락 상실 |




# 특화 튜닝 모델 확인 (Text-to-SQL)

## 11. 개요

[Task]
- 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 응답을 직접 실행하여 확인
- baseline과의 실행 비교 없이, 특화 모델 하나의 출력만 확인

[Model]
- 특화 튜닝: SLM-SQL-0.5B (Qwen2.5-Coder-0.5B-Instruct 기반, SFT + RL 튜닝)

[Note: Model Description]

 - Qwen2.5-Coder-0.5B-Instruct(범용 코드 모델)를 Text-to-SQL 작업에 SFT + RL로 추가 튜닝한 모델
- BIRD 벤치마크 기준, 같은 0.5B 파라미터에서도 튜닝 전후 정확도 차이가 크게 발생

| 모델 상태 | Execution Accuracy |
|---|---|
| 튜닝 전 (일반 SFT만, Qwen2.5-Coder-0.5B-Instruct) | 42.13% |
| 튜닝 후 (SFT + RL, SLM-SQL-0.5B) | 65.31% |

- (출처: SLM-SQL 논문, arXiv:2507.22478)


## 12. SQL 모델 로드

In [13]:
sql_tuned_id = "cycloneboy/SLM-SQL-0.5B"

sql_tuned_tokenizer = AutoTokenizer.from_pretrained(sql_tuned_id)
sql_tuned_model = AutoModelForCausalLM.from_pretrained(
    sql_tuned_id, torch_dtype=compute_dtype, device_map="auto"
)

print("특화 SQL 모델 로드 완료:", sql_tuned_id)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.33k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

특화 SQL 모델 로드 완료: cycloneboy/SLM-SQL-0.5B


## 13. SQL 생성 프롬프트 정의

- 스키마 정보를 함께 제공하고, 자연어 질문을 SQL로 변환

In [17]:
sql_schema = """
employees(id, name, dept_id, salary)
departments(dept_id, dept_name, location)
"""
sql_question = "서울에 위치한 부서별 평균 급여를 높은 순으로 보여줘."

sql_prompt = f"""다음 스키마를 참고하여 질문에 맞는 SQL 쿼리만 작성하세요.

스키마: {sql_schema}
질문: {sql_question}
SQL:"""

print(sql_prompt)

다음 스키마를 참고하여 질문에 맞는 SQL 쿼리만 작성하세요.

스키마: 
employees(id, name, dept_id, salary)
departments(dept_id, dept_name, location)

질문: 서울에 위치한 부서별 평균 급여를 높은 순으로 보여줘.
SQL:


## 14. SQL 생성 실행

In [15]:
def generate_sql(tokenizer, model, prompt: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


tuned_sql = generate_sql(sql_tuned_tokenizer, sql_tuned_model, sql_prompt)

print("=== 특화 튜닝 모델: SLM-SQL-0.5B ===")
print(tuned_sql)

=== 특화 튜닝 모델: SLM-SQL-0.5B ===
SELECT 
    d.dept_name,
    AVG(e.salary) AS average_salary
FROM 
    employees e
INNER JOIN 
    departments d ON e.dept_id = d.dept_id
WHERE 
    d.location = '서울'
GROUP BY 
    d.dept_name
ORDER BY 
    average_salary DESC;


# [로컬 전용 실습] 네트워크 연결 없이 실행하기

## 15. 오프라인 추론 확인

[Task]
- 위 셀들을 모두 실행하여 모델 다운로드가 완전히 끝난 것을 확인한 뒤 진행
- Wi-Fi 또는 랜선을 분리한 상태에서 아래 셀을 실행하여, 로컬에 저장된 sLLM이 인터넷 없이도 동작하는지 확인

[Note]
- Colab은 클라우드 가상 환경이라 이 실습을 수행할 수 없음 (로컬 환경 전용)
- Colab에서는 (네트워크가 연결되어 있다면) "캐시 재사용이 잘 되는지"만 확인됨
- local_files_only=True 옵션으로 캐시된 파일만 사용하도록 강제

In [16]:
# 네트워크를 분리한 뒤 이 셀을 실행
offline_tokenizer = AutoTokenizer.from_pretrained(
    instruct_model_id, local_files_only=True
)
offline_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    torch_dtype=compute_dtype,
    device_map="auto",
    local_files_only=True
)

messages = [{"role": "user", "content": "sLLM이 무엇인지 한 문장으로 설명해줘"}]
inputs = offline_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(offline_model.device)
output_ids = offline_model.generate(**inputs, max_new_tokens=100, do_sample=False)

print(offline_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

SLLM은 "사람과 로봇의 합성 언어 모델"을 의미합니다.


# sLLM 실행 및 비교
## 필수
### Q. 단일턴 비교와 멀티턴 대화에서 관찰한 차이중 가장 인상적이었던 차이는 무엇이었고 그로부터 어떤 시사점을 얻었나요? (범용 LLM vs Base vs Instruct 비교표 첨부할 것)

| 프롬프트 | 범용 LLM | sLLM-Base | sLLM-Instruct |
|---|---|---|---|
| 1. 사실 질문 | 세종대왕이 만든 문자는 훈민정음입니다. 오늘날의 한글을 뜻합니다. | 정답을 조선자유자치라는 할루시네이션 발생한 듯함, 문장이 중간에 끊기거나 반복됨 | 정답(O), 간결하고 핵심적인 답변 |
| 2. 지시문형 | 인공지능 발전으로 산업 자동화가 가속화되며, 특히 제조업과 금융업에서 변화가 두드러지고 있다. | 지시 무시, 프롬프트 텍스트를 이어쓰는 경향이 보임 | 지시사항 대부분 준수하여 응답함|
| 3. 추론 문제 | 안경을 쓰지 않은 학생은 (30-12=18)명입니다.그중 남학생이 10명이므로, 안경을 쓰지 않은 여학생은 18-10=8명입니다. | 추론 실패, 실제 정답과 다른 답을 발생함| 기본 추론 가능, 간단한 연산이라 답을 제시함 |
| 4. 멀티턴 | 렌터카 기준 제주 동부 1일 코스를 추천합니다.비자림(오전) → 성산일출봉 → 성산에서 점심 → 섭지코지 → 광치기해변(노을)숲·오름·해안을 하루에 고르게 즐기면서 이동 동선도 비교적 짧습니다. 비가 많이 오면 성산일출봉 대신 아쿠아플라넷 같은 실내 관광지로 바꾸세요.방문 전 운영·기상 정보는 비짓제주 공식 관광 가이드에서 확인하는 것이 좋습니다. | 이전 대화의 맥락을 파악하지 못하고 입력된 단순 문장만 무한 반복 | 단기 문맥 유지하여 응답함 1박 2일 일정을 제시해줌|

시사점 : 확실히 파인 튜닝이 필요하다 instruct 모델의 경우, base 보다 유의미하게 문장을 생성한다. 범용 LLM의 경우 상당히 안정적이고 유용한 정보를 생성한다.



### Q. 특화 SQL 모델을 관찰하고 "왜 특화 튜닝이 필요한가" 에 답해보고, Day2 파인튜닝 실습에서 확인하고 싶은 것은 무엇인가요?

1. 실습에 사용한 모델은 파라미터가 작은 규모이지만 복잡한 자연어를 생성했다. 그 이유는 파인튜닝으로 확실한 학습 포인트를 잡고 학습시킨다면 대규모 모델만큼 성능을 볼 수 있음을 확인했다.
2. 도메인마다 다른 학습 능력을 보일 것 같은데 어떤 부분에서 파인 튜닝이 어려운지 알고 싶다. 가능하다면 변화 과정을 세밀하게 비교해보면서 파라미터를 비교해보고 싶다.

## (선택 - 다음중 하나의 질문에 답하시오)
### Q1. 범용 LLM이 sLLM-Instruct보다 나은 답을 냈다면, 그 이유를 "모델 크기"라고 단정할 수 있을까요? 단정할 수 없다면 왜인지, 또 어떻게 하면 "크기 효과"만 따로 확인할 수 있을지 설명해보세요.

- 모델 크기또한 중요하지만 강화 학습을 병행하여 더욱 답을 안정적으로 제시할 수 있는 것 같다. 즉 모델 크기가 클수록 용량이 크기에 질문 정확도가 증가한 것 같다. 동일 계열(Family) 모델 라인업 비교을 통해 비교 가능해보인다. 동일 질문 응답 결과를 보며 성공률을 비교해보는 방식이 가능해보인다.

### Q2. SLM-SQL-0.5B에게 방금 전 단일턴 비교에서 썼던 프롬프트(사실질문/추론문제)를 그대로 넣어보세요. 특화 튜닝이 SQL 성능을 올린 대신 무엇을 잃었을까요?


### Q3. 여러분이 회사에서 '고객 문의에 SQL을 자동 생성해주는 기능'을 만든다면, 오늘 본 세 가지(범용 LLM API, sLLM Instruct, 특화 튜닝 모델) 중 무엇을 쓰겠습니까? 비용/속도/정확도 관점에서 설명해보세요.

우선 파인 튜닝한 sLLM Instruct 모델을 사용할 것이다.
비용 관점에서는 범용 LLM API의 경우 모델 사용량이 큰 과제가 아니기에 큰 금액이 나오진 않겠지만 충분히 sLLM을 구축하여 만들 수 있는 문제 난이도로 보이기 때문이다. 단지 정확도 측면에서는 범용 LLM의 경우가 모델 크기 측면에서 강점이 있어 특화 튜닝을 통한 정확도 비교를 거친 후 판단하면 좋을 것 같다.
